# BB84 on real hardware: telling noise apart from an eavesdropper

| | |
|---|---|
| **Level** | Intermediate |
| **Time** | About 60 minutes |
| **Prerequisites** | The BB84 protocol; binary entropy |
| **Default devices** | Rigetti Cepheus and IQM Garnet |
| **Hardware jobs** | 6 on Garnet, 1 on Rigetti |
| **Approximate cost** | about 1,050 credits on Garnet at 1000 shots; about 10 credits on Rigetti, which is billed by execution time |
| **Hardware notes** | Tested 25 September 2026 at 100 shots. On IQM Garnet, run verbatim, the QBER was 1.75% with no channel and 2.0% with 16 repetitions: its single-qubit gates are good enough that the channel adds little. On Rigetti Cepheus, with no channel, it was 6.75%. |

Shot counts for the hardware runs are set at the end of the **Setup** cell. The devices are set in the first hardware cell. Credits are charged only when a hardware cell runs.

*QUEST intermediate and advanced series: Cryptography and Security*

BB84 is usually taught as a protocol that detects eavesdropping. Eve measures the qubits in transit, her measurements disturb them, and Alice and Bob notice by comparing part of their key. On an ideal simulator the quantum bit error rate (QBER) is zero without Eve and 25% with her.

Real hardware always has some error. Gate error, decoherence and readout error produce the same signature as an eavesdropper, and the protocol cannot tell them apart. To stay secure, Alice and Bob must attribute every error to Eve and discard key material to compensate. This notebook measures what that costs on current devices.

We implement BB84, add an intercept-resend eavesdropper, derive the 11% abort threshold, and then run the protocol on real devices with no eavesdropper present. Making the quantum channel longer shows how the device's own noise eats into the key.

**Learning objectives**

1. Implement BB84, including basis reconciliation, sifting and QBER estimation.
2. Model an intercept-resend eavesdropper and predict the QBER she causes.
3. Derive the 11% threshold from the Shor-Preskill key rate.
4. Measure the QBER of real devices running the protocol with no eavesdropper.
5. Convert a measured QBER into a secure key rate.
6. Explain why deployed QKD uses photons rather than gate-based processors.

**Background needed:** single-qubit states, and measurement in the $Z$ and $X$ bases. No cryptography background is assumed. The security argument is sketched, not proved.


## The protocol

Alice picks a random bit $a$ and a random basis, and encodes the bit:

| Basis | $a = 0$ | $a = 1$ |
|---|---|---|
| $Z$ | $\lvert 0 \rangle$ | $\lvert 1 \rangle$ |
| $X$ | $\lvert + \rangle$ | $\lvert - \rangle$ |

Bob measures each qubit in a basis he picks at random. Afterwards they announce their bases, never their bits, on a public channel, and discard every round where the bases differ. This step is called **sifting** and keeps about half the rounds.

In the remaining rounds Bob measured in Alice's basis, so on a noiseless channel his bit equals hers. The **quantum bit error rate** is the fraction of sifted rounds where they disagree:

$$Q = \frac{\#\{i \in \text{sifted} : b_i \neq a_i\}}{\#\{\text{sifted}\}}$$

Alice and Bob estimate $Q$ by comparing a random subset of their sifted bits in public, then discard those bits. BB84's security claim is that $Q$ limits how much Eve can know about the rest.

This rests on two properties of the four states. They form two mutually unbiased bases, so measuring a $Z$-basis state in the $X$ basis gives a random result and destroys the encoded bit. And no single measurement distinguishes all four, so Eve must guess a basis and risks guessing wrong.

## Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import brentq

# Qiskit for circuit construction and local simulation
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

# qBraid for unified device access
from qbraid.runtime import QbraidProvider

# Housekeeping
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 110

sim = AerSimulator(seed_simulator=42)

print("Setup complete.")

# Shot counts for the hardware runs. More shots reduce statistical error but cost
# more on most devices; the README lists prices.
SHOTS = 1000
QUEST_JOB_TAGS = {"quest": "crypto-bb84"}   # labels this notebook's hardware jobs for QUEST usage statistics


## One round of the protocol

Each round uses one qubit. Alice applies an optional $X$ gate to set the bit, then an optional $H$ to move to the $X$ basis. Bob applies an optional $H$ to rotate his basis onto the computational basis, then measures.

We write `0` for the $Z$ basis and `1` for the $X$ basis. As elsewhere in Qiskit, qubit 0 is the rightmost bit of a returned bitstring.

The `channel_reps` argument adds pairs of $X$ gates to represent time in transit. Each pair is logically the identity, so the ideal result does not change, but on hardware every gate adds a little error. This lengthens the channel without changing what the protocol should return.

In [ ]:
Z_BASIS, X_BASIS = 0, 1


def bb84_round(alice_bit, alice_basis, bob_basis, channel_reps=0):
    """One BB84 round on a single qubit."""
    qc = QuantumCircuit(1, 1)

    # Alice prepares
    if alice_bit:
        qc.x(0)
    if alice_basis == X_BASIS:
        qc.h(0)
    qc.barrier()

    # The channel: channel_reps identity-equivalent X pairs
    for _ in range(channel_reps):
        qc.x(0)
        qc.barrier()
        qc.x(0)
        qc.barrier()

    # Bob measures in his chosen basis
    if bob_basis == X_BASIS:
        qc.h(0)
    qc.measure(0, 0)
    return qc


bb84_round(alice_bit=1, alice_basis=X_BASIS, bob_basis=X_BASIS, channel_reps=1).draw('mpl')

The barriers matter. Without them Qiskit's transpiler recognises that $X \cdot X = I$ and deletes the channel. That is correct logically, but it removes the gates we want the hardware to run.

Barriers only instruct Qiskit, though. After qBraid submits a circuit, the device's own compiler processes it again, and in our tests it removed the channel on both IQM Garnet and Rigetti Cepheus, barriers included. The hardware section below shows how this notebook gets the channel onto the device, and how to check what the device actually ran.

## Running the protocol without an eavesdropper

Alice generates random bits and bases, Bob generates random bases, and we run every round. Eve is built into the same function so we can switch her on later: she intercepts a fraction `eve_fraction` of the rounds, measures each in a basis she chooses at random, and forwards a fresh qubit prepared from her outcome in her own basis.

In [ ]:
def run_protocol(n_rounds, channel_reps=0, eve_fraction=0.0, backend=None, seed=None):
    """
    Run n_rounds of BB84 and return the sifted keys and the QBER.

    eve_fraction is the fraction of rounds an intercept-resend eavesdropper touches.
    backend defaults to the ideal Aer simulator.
    """
    rng = np.random.default_rng(seed)
    backend = sim if backend is None else backend

    alice_bits = rng.integers(0, 2, n_rounds)
    alice_bases = rng.integers(0, 2, n_rounds)
    bob_bases = rng.integers(0, 2, n_rounds)

    # Eve intercepts a random subset of rounds, measuring in a random basis
    intercepted = rng.random(n_rounds) < eve_fraction
    eve_bases = rng.integers(0, 2, n_rounds)

    sent_bits, sent_bases = alice_bits.copy(), alice_bases.copy()
    for i in np.flatnonzero(intercepted):
        if eve_bases[i] == alice_bases[i]:
            continue                          # right basis: she learns the bit, state passes intact
        sent_bits[i] = rng.integers(0, 2)     # wrong basis: her outcome is uniformly random
        sent_bases[i] = eve_bases[i]          # and she resends in her own basis

    circuits = [bb84_round(sent_bits[i], sent_bases[i], bob_bases[i], channel_reps)
                for i in range(n_rounds)]
    counts = backend.run(circuits, shots=1).result().get_counts()
    bob_bits = np.array([int(next(iter(c))) for c in counts])

    sifted = alice_bases == bob_bases
    qber = float((bob_bits[sifted] != alice_bits[sifted]).mean())
    return {'alice_key': alice_bits[sifted], 'bob_key': bob_bits[sifted],
            'n_sifted': int(sifted.sum()), 'qber': qber}

In [ ]:
N_ROUNDS = 4000

clean = run_protocol(N_ROUNDS, seed=1)
alice_str = ''.join(map(str, clean['alice_key'][:40]))
bob_str = ''.join(map(str, clean['bob_key'][:40]))

print(f"Rounds run:        {N_ROUNDS}")
print(f"Sifted key length: {clean['n_sifted']}  ({clean['n_sifted'] / N_ROUNDS:.1%} of rounds)")
print(f"QBER:              {clean['qber']:.4f}")
print(f"Alice, first 40 sifted bits: {alice_str}")
print(f"Bob,   first 40 sifted bits: {bob_str}")

Sifting keeps close to half the rounds, as expected when two independent fair coins must agree. On the ideal simulator the QBER is zero and the two sifted keys are identical. This is the baseline for everything that follows.

## Adding an eavesdropper

In an intercept-resend attack, Eve captures each qubit, measures it in a basis she picks at random, and sends Bob a new qubit prepared from her result in her basis.

Half the time she picks Alice's basis. Her result is then certain, she learns the bit, and she forwards the same state Alice sent. Those rounds show no disturbance.

The other half, she picks the other basis. Her result is random and tells her nothing, and she forwards a state in the wrong basis. When such a round survives sifting, Bob's result is random and disagrees with Alice half the time.

So an Eve who intercepts every qubit causes

$$Q_{\text{Eve}} = \underbrace{\tfrac{1}{2}}_{\text{wrong basis}} \times \underbrace{\tfrac{1}{2}}_{\text{random result}} = \tfrac{1}{4},$$

and one who intercepts a fraction $f$ of the rounds causes $Q = f/4$. Because $Q$ grows in proportion to $f$, the QBER tells Alice and Bob how much of the traffic Eve touched.

In [ ]:
fractions = np.linspace(0, 1, 11)
eve_qbers = [run_protocol(3000, eve_fraction=f, seed=100 + i)['qber']
             for i, f in enumerate(fractions)]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(fractions, fractions / 4, 'k--', linewidth=2, alpha=0.6, label='Prediction $Q = f/4$')
ax.plot(fractions, eve_qbers, 'o-', color='#a02580', markersize=9, linewidth=2,
        markeredgecolor='white', markeredgewidth=1.2, label='Simulated')
ax.set_xlabel('Fraction of rounds Eve intercepts, $f$')
ax.set_ylabel('QBER')
ax.set_ylim(0, 0.3)
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(f"QBER at f = 1: {eve_qbers[-1]:.4f}  (predicted 0.2500)")

## Where the 11% comes from

BB84 is usually said to abort above a QBER of about 11%. Deriving the number shows why hardware noise has a cost.

After sifting, Alice's and Bob's strings are similar but not identical, and Eve knows part of them. Error correction makes the strings match, which leaks some information to Eve. Privacy amplification then compresses the result into a shorter string that Eve knows almost nothing about. For BB84 with one-way post-processing, the Shor-Preskill analysis gives an asymptotic secure key rate per sifted bit of

$$r(Q) = 1 - 2h(Q), \qquad h(Q) = -Q\log_2 Q - (1 - Q)\log_2 (1 - Q).$$

One factor of $h(Q)$ pays for error correction and the other bounds Eve's information. The rate reaches zero when $h(Q) = 1/2$. That equation has no closed-form solution, so we solve it numerically.

In [ ]:
def binary_entropy(q):
    """Binary entropy in bits, with h(0) = h(1) = 0."""
    q = np.atleast_1d(np.asarray(q, dtype=float))
    out = np.zeros_like(q)
    m = (q > 0) & (q < 1)
    out[m] = -q[m] * np.log2(q[m]) - (1 - q[m]) * np.log2(1 - q[m])
    return out


def key_rate(q):
    """Asymptotic Shor-Preskill secure key rate per sifted bit."""
    return np.maximum(0.0, 1 - 2 * binary_entropy(q))


q_threshold = brentq(lambda q: 1 - 2 * binary_entropy(q)[0], 1e-9, 0.5 - 1e-9)

print(f"Zero-rate threshold:              Q* = {q_threshold:.4%}")
print(f"Eve interception fraction at Q*:   f = {4 * q_threshold:.3f}")
for q in [0.01, 0.02, 0.05, 0.08]:
    print(f"  Q = {q:.0%}  ->  r = {key_rate(q)[0]:.3f}")

In [ ]:
q_grid = np.linspace(0, 0.2, 400)

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.plot(q_grid, key_rate(q_grid), color='#1a5285', linewidth=2.5)
ax.fill_between(q_grid, 0, key_rate(q_grid), color='#1a5285', alpha=0.12)
ax.axvline(q_threshold, color='k', linestyle='--', alpha=0.6)
ax.annotate(f'$Q^*$ = {q_threshold:.2%}',
            xy=(q_threshold, 0.0), xytext=(q_threshold + 0.012, 0.28), fontsize=11,
            arrowprops=dict(arrowstyle='->', alpha=0.6))
ax.set_xlabel('QBER')
ax.set_ylabel('Secure key rate per sifted bit, $r$')
ax.set_title('Shor-Preskill key rate for BB84')
ax.set_xlim(0, 0.2)
ax.set_ylim(0, 1.02)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Two points about this curve.

The threshold is $Q^* = 11.00\%$. Since $Q = f/4$, that corresponds to Eve intercepting 44% of the rounds. The protocol keeps producing key while privacy amplification can remove what she knows. In that sense BB84 bounds Eve's knowledge; it does not simply detect her.

The rate also falls steeply well before the threshold: about 0.72 at $Q = 2\%$ and 0.43 at $Q = 5\%$. Errors are costly long before they stop the protocol, and a real device produces them whether or not anyone is listening.

## Restructuring the protocol for hardware

The implementation above submits one circuit per round. On hardware that would mean thousands of jobs.

Sifting and basis reconciliation are classical bookkeeping. The only thing the device determines is the probability that Bob's result differs from Alice's bit in a sifted round, and that depends on which of the four states Alice sent. So we prepare all four BB84 states at once on four qubits, measure each in its matching basis, and read the error rate from the results. One circuit then gives statistics for every state at once.

In [ ]:
BB84_STATES = [(0, Z_BASIS), (1, Z_BASIS), (0, X_BASIS), (1, X_BASIS)]
STATE_LABELS = ['|0>', '|1>', '|+>', '|->']


def sifted_error_circuit(channel_reps=0):
    """Prepare all four BB84 states in parallel, one per qubit, each measured in its own basis."""
    n = len(BB84_STATES)
    qc = QuantumCircuit(n, n)

    for q, (bit, basis) in enumerate(BB84_STATES):
        if bit:
            qc.x(q)
        if basis == X_BASIS:
            qc.h(q)
    qc.barrier()

    for _ in range(channel_reps):
        qc.x(range(n))
        qc.barrier()
        qc.x(range(n))
        qc.barrier()

    for q, (bit, basis) in enumerate(BB84_STATES):
        if basis == X_BASIS:
            qc.h(q)
    qc.measure(range(n), range(n))
    return qc


def per_state_error_rates(counts, shots):
    """
    Error rate for each of the four BB84 states from a packed-circuit shot record.
    Qiskit puts the highest-indexed clbit leftmost, so clbit q is bitstring[n - 1 - q].
    """
    n = len(BB84_STATES)
    errors = np.zeros(n)
    for bitstring, num in counts.items():
        b = bitstring.replace(' ', '')
        for q, (bit, _) in enumerate(BB84_STATES):
            if int(b[n - 1 - q]) != bit:
                errors[q] += num
    return errors / shots


def qber_from_counts(counts, shots):
    """QBER averaged over the four BB84 states."""
    return float(per_state_error_rates(counts, shots).mean())


sifted_error_circuit(channel_reps=1).draw('mpl', fold=90)

In [ ]:
# Confirm the packed circuit reproduces the ideal result before spending credits
for reps in [0, 4, 16]:
    counts = sim.run(sifted_error_circuit(reps), shots=4000).result().get_counts()
    rates = per_state_error_rates(counts, 4000)
    print(f"channel_reps = {reps:2d}   ideal QBER = {rates.mean():.4f}   per state = {rates}")

The ideal QBER stays at zero for every channel length, because the channel is built from identity pairs. Any error a real device reports from here on comes from the hardware.

## Running on real devices

BB84 needs only single-qubit gates and measurement, so the qubit layout does not matter here. What matters is the error of single-qubit gates and of measurement. By default the notebook runs on two superconducting devices. You can add AQT's trapped-ion device by uncommenting its line in the next cell; it runs in scheduled windows and costs more per shot.

| Device | Vendor | Type | Main error sources |
|---|---|---|---|
| `rigetti:rigetti:qpu:cepheus-1-108q` | Rigetti | Superconducting, 107 qubits | Single-qubit gate error, readout error, $T_1/T_2$ |
| `aws:iqm:qpu:garnet` | IQM | Superconducting, 20 qubits | Single-qubit gate error, readout error, $T_1/T_2$ |
| `aws:aqt:qpu:ibex-q1` (optional) | AQT | Trapped ion, 12 qubits | Single-qubit gate error, readout error, slower gates |

**Keeping the channel on the device.** On IQM Garnet the notebook submits a *verbatim* program: the circuit translated into Garnet's native gates on physical qubits we choose, which the device runs gate for gate. Rigetti offers no such mode through qBraid, and its compiler removes the identity pairs, so on Rigetti the notebook runs only the circuit without a channel (`channel_reps = 0`). After every job, `check_what_ran` prints how many gates were sent and how many the device ran.

This submits 6 jobs on Garnet and 1 on Rigetti. Depending on queues, the next cell may take minutes to hours.

In [ ]:
provider = QbraidProvider()

# Devices are named by qBraid QRN. The README lists devices, prices and availability.
BACKENDS = {
    'Rigetti Cepheus': 'rigetti:rigetti:qpu:cepheus-1-108q',
    'IQM Garnet':      'aws:iqm:qpu:garnet',
    # 'AQT IBEX Q1':   'aws:aqt:qpu:ibex-q1',   # trapped ion; runs in scheduled windows; 2.35 credits per shot
}

COLORS = {
    'Rigetti Cepheus': '#c63792',
    'IQM Garnet':      '#1a5285',
    'AQT IBEX Q1':     '#2d7a4f',
}

CHANNEL_REPS = [0, 1, 2, 4, 8, 16]

devices = {name: provider.get_device(dev_id) for name, dev_id in BACKENDS.items()}
print("Configured backends:", list(devices))

import re

_NOT_GATES = {"openqasm", "include", "bit", "qubit", "box", "measure", "declare", "pragma",
              "b", "c", "meas", "ro", "barrier", "fence", "delay", "halt", "reset", "defcal", "cal"}

def _count_gates(program_text):
    """(all gates, two-qubit gates) in a compiled OpenQASM or Quil program."""
    total = two = 0
    for line in program_text.splitlines():
        m = re.match(r"\s*([A-Za-z_]+)", line)
        if not m or m.group(1).lower() in _NOT_GATES:
            continue
        total += 1
        if m.group(1).lower() in ("cz", "cx", "cnot", "iswap", "xy", "cphase", "ecr", "ms", "zz"):
            two += 1
    return total, two

def check_what_ran(job, circuit):
    """Compare the circuit we sent with the program the device actually ran.

    Device compilers rewrite circuits before running them. Usually that only
    changes the gate names, but a compiler can also remove gates that cancel,
    such as a circuit followed by its inverse. This prints both gate counts.
    """
    ops = [inst.operation for inst in circuit.data if inst.operation.name not in ("barrier", "measure")]
    sent_total, sent_two = len(ops), sum(1 for op in ops if op.num_qubits == 2)
    try:
        program = job.client.get_job_compiled_program(job.id)
    except Exception as err:
        print(f"  could not fetch the compiled program ({type(err).__name__}); check skipped")
        return None
    ran_total, ran_two = _count_gates(getattr(program, "data", str(program)))
    print(f"  gates sent {sent_total} ({sent_two} two-qubit); device ran {ran_total} ({ran_two} two-qubit)")
    removed = (sent_two and ran_two < 0.5 * sent_two) or (sent_total >= 10 and ran_total < 0.5 * sent_total)
    if removed:
        print("  WARNING: the compiler removed most of the gates. "
              "This result does not measure the circuit you built.")
    return not removed


# ---- Running circuits exactly as written ----------------------------------
# A device's compiler removes gates that cancel, such as a circuit followed by
# its own inverse, even across barriers. For this notebook that would erase the
# experiment. IQM Garnet (through Amazon Braket) accepts "verbatim" programs:
# native gates on named physical qubits, run gate for gate. run_exactly() uses
# that on Garnet. Rigetti has no such mode through qBraid, so there it submits
# normally, and check_what_ran() reports whether the circuit survived.
from qiskit import transpile
from qiskit.transpiler import CouplingMap
from qbraid_core.services.runtime.schemas import Program

GARNET_EDGES = [(3, 4), (3, 8), (4, 5), (8, 9), (8, 13), (9, 10), (9, 14), (10, 11), (10, 15),
                (11, 12), (11, 16), (12, 17), (13, 14), (14, 15), (14, 18), (15, 16), (15, 19),
                (16, 17), (16, 20), (18, 19), (19, 20)]
GARNET_PATH = [14, 15, 16, 17, 12, 11, 10, 9]    # eight connected qubits in a line


def runs_exactly(device_id):
    """True for devices where run_exactly() can bypass the compiler."""
    return device_id.startswith('aws:iqm:')


def garnet_verbatim(qc, physical=GARNET_PATH):
    """qc in IQM's native gates (prx, cz) on the given physical qubits, as a verbatim program."""
    phys = list(physical)[:qc.num_qubits]
    index = {p: i for i, p in enumerate(phys)}
    links = [(index[a], index[b]) for a, b in GARNET_EDGES if a in index and b in index]
    cmap = CouplingMap(links + [(b, a) for a, b in links])
    # optimization_level=0 translates gates without cancelling or merging any.
    native = transpile(qc, basis_gates=['r', 'cz'], coupling_map=cmap,
                       initial_layout=list(range(len(phys))), optimization_level=0,
                       seed_transpiler=1)
    body, reads = [], []
    for inst in native.data:
        qs = [phys[native.find_bit(q).index] for q in inst.qubits]
        name = inst.operation.name
        if name == 'r':
            theta, phi = (float(p) for p in inst.operation.params)
            body.append(f"prx({theta:.12f}, {phi:.12f}) ${qs[0]};")
        elif name == 'cz':
            body.append(f"cz ${qs[0]}, ${qs[1]};")
        elif name == 'measure':
            reads.append(f"b[{native.find_bit(inst.clbits[0]).index}] = measure ${qs[0]};")
        elif name != 'barrier':
            raise ValueError(f"unexpected gate after translation: {name}")
    return (f"OPENQASM 3.0;\nbit[{qc.num_clbits}] b;\n#pragma braket verbatim\nbox{{\n"
            + "\n".join(body) + "\n}\n" + "\n".join(reads) + "\n")


def run_exactly(name, device, qc, shots, physical=GARNET_PATH):
    """Submit qc; return (counts, ran_as_written). On Garnet it runs gate for gate."""
    if runs_exactly(BACKENDS[name]):
        job = device.submit(Program(format='qasm3', data=garnet_verbatim(qc, physical)),
                            shots=shots, tags=QUEST_JOB_TAGS)
    else:
        job = device.run(qc, shots=shots, tags=QUEST_JOB_TAGS)
    counts = job.result().data.get_counts()
    ran_as_written = check_what_ran(job, qc)
    return counts, ran_as_written is not False


In [ ]:
hw_qber = {name: {} for name in BACKENDS}
hw_per_state = {name: {} for name in BACKENDS}

for name, device in devices.items():
    # Without a verbatim mode the compiler deletes the channel, so only reps = 0 is meaningful.
    reps_to_run = CHANNEL_REPS if runs_exactly(BACKENDS[name]) else [0]
    for reps in reps_to_run:
        qc = sifted_error_circuit(reps)
        counts, ran_as_written = run_exactly(name, device, qc, SHOTS)
        if not ran_as_written:
            print(f"{name:18s} channel_reps={reps:2d}  not recorded: the compiler removed the channel")
            continue
        hw_qber[name][reps] = qber_from_counts(counts, SHOTS)
        hw_per_state[name][reps] = per_state_error_rates(counts, SHOTS)
        print(f"{name:18s} channel_reps={reps:2d}  QBER = {hw_qber[name][reps]:.4f}")


## QBER against channel length

The ideal curve is flat at zero. The devices sit above it, with no eavesdropper anywhere in the experiment.

In [ ]:
fig, ax = plt.subplots(figsize=(9.5, 6))

ax.axhline(0, color='k', linestyle='--', linewidth=2, alpha=0.5, label='Ideal (no Eve)')
ax.axhline(q_threshold, color='#8a1c1c', linestyle='-.', linewidth=2,
           label=f'Abort threshold $Q^*$ = {q_threshold:.1%}')
ax.axhline(0.25, color='gray', linestyle=':', linewidth=2,
           label='Fully intercepting Eve (25%)')

for name in BACKENDS:
    reps = sorted(hw_qber[name])
    ax.plot(reps, [hw_qber[name][r] for r in reps], 'o-', color=COLORS[name], label=name,
            markersize=11, linewidth=2.5, markeredgecolor='white', markeredgewidth=1.5)

ax.set_xlabel('Channel length (identity-pair repetitions)')
ax.set_ylabel('QBER')
ax.set_title('BB84 error rate on real hardware, with no eavesdropper present')
ax.set_ylim(-0.01, 0.3)
ax.grid(alpha=0.3)
ax.legend(loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()

At `channel_reps = 0` the circuit prepares a state and measures it straight away, so the QBER there is mostly readout error plus a gate or two. A few percent is typical, and it already costs key: at $Q = 3\%$ the rate is about 0.61, so nearly 40% of the sifted key is spent defending against an eavesdropper who is not there.

How the QBER changes with channel length depends on the device's single-qubit gate error and coherence time. In our tests on IQM Garnet, run verbatim, the QBER rose only from 1.75% to 2.0% over 16 repetitions (128 extra gates), which is within the statistical error at 100 shots: its single-qubit gates are good enough that even a long channel adds little. On Rigetti Cepheus the QBER with no channel was 6.75%, mostly from readout error. If it passes $Q^*$, the protocol yields no secure key and Alice and Bob must abort, even though the only thing in the channel is the device's own noise.

## What the noise costs

Each measured QBER converts to a key rate. The last columns show how many secure bits survive, and what level of eavesdropping the observed error rate is indistinguishable from.

In [ ]:
rows = []
for name in BACKENDS:
    for reps in sorted(hw_qber[name]):
        q = hw_qber[name][reps]
        r = float(key_rate(q)[0])
        rows.append({
            'Backend': name,
            'Channel reps': reps,
            'QBER': f'{q:.4f}',
            'Key rate r(Q)': f'{r:.3f}',
            'Secure bits per 1000 sifted': f'{1000 * r:.0f}',
            'Equivalent Eve fraction': f'{min(4 * q, 1.0):.2f}',
            'Verdict': 'ABORT' if q >= q_threshold else 'ok',
        })

pd.DataFrame(rows).set_index(['Backend', 'Channel reps'])

Read the "equivalent Eve fraction" column this way: a device with a QBER of 6% looks, from inside the protocol, exactly like a noiseless device with an eavesdropper intercepting a quarter of the traffic. Alice and Bob cannot tell which they have, so they must assume the second.

## Which states are noisiest

The four states are not equally robust. $\lvert 0 \rangle$ needs no gate before measurement, $\lvert 1 \rangle$ needs one $X$, and the two $X$-basis states need an $H$ at each end. Readout error on superconducting devices is also usually asymmetric: a qubit in $\lvert 1 \rangle$ can decay to $\lvert 0 \rangle$ during the measurement, but not the other way round.

In [ ]:
per_state_rows = []
for name in BACKENDS:
    rates = hw_per_state[name][0]
    row = {'Backend': name}
    row.update({lbl: f'{rate:.4f}' for lbl, rate in zip(STATE_LABELS, rates)})
    row['Mean'] = f'{rates.mean():.4f}'
    per_state_rows.append(row)

pd.DataFrame(per_state_rows).set_index('Backend')

If the $\lvert 1 \rangle$ column is consistently worse than $\lvert 0 \rangle$, the cause is asymmetric readout error, which can largely be corrected in classical post-processing. If the two $X$-basis columns are worse than both $Z$-basis columns, the cause is the extra $H$ gates. Telling the two apart shows whether readout or gate calibration needs attention. The benchmarking notebook in this series takes this diagnosis further.

## Why deployed QKD looks different

This experiment implements BB84's logic and error accounting. It is not a QKD system, for three reasons.

The qubit never leaves the chip. In a real link the quantum state travels a physical distance, which is what gives Eve something to intercept. Here Alice, Bob and Eve are all the same processor, and the channel is a few gates on one qubit.

Deployed QKD sends photons through fibre or free space, where the main problem is loss. A photon that never arrives is a missing round, which does not raise the QBER. BB84 tolerates a great deal of loss but very little error, and the two are handled by different parts of the analysis.

Real systems also face attacks with no equivalent on a gate-based processor. A laser pulse attenuated to one photon on average sometimes contains two, and Eve can keep one and forward the other without disturbing anything. Decoy-state protocols were developed to bound that attack. Attacks on Bob's detectors led to measurement-device-independent QKD.

What the experiment does show is the central point of the security argument. The protocol has one observable, $Q$, and cannot ask where the errors came from, so every error is charged to Eve. On current processors that charge uses up a large part of the key even with no channel at all.

## Going further

- **Estimate $Q$ from a sample.** Alice and Bob compare only a random subset of their bits. Estimate $Q$ from 10% of the sifted bits and put a confidence interval on it. How many rounds do you need before a decision near the threshold is trustworthy?
- **Add finite-key effects.** The rate $r = 1 - 2h(Q)$ assumes very long keys. Compare it with a finite-key bound for blocks of $10^4$ and $10^6$ bits.
- **Let the compiler remove the channel.** Remove the barriers and use `optimization_level=3`. The QBER should drop to its `channel_reps = 0` value at every length, because the compiler deletes the identity pairs.
- **Build E91.** Distribute entangled pairs, measure in random bases, and use a CHSH violation as the security test. Compare what the two protocols need from the hardware.
- **Reconcile the measured keys.** Implement the Cascade error-correction protocol on the sifted keys and compare its information leakage with the $h(Q)$ bound.